# OTP-FM: Exact Gaussian Solutions

This notebook demonstrates the analytical (exact) solutions for OTP-FM with Gaussian marginals.

Under the Gaussian ansatz, the multi-marginal optimal transport problem with intermediate marginal constraints admits closed-form solutions. This provides ground truth for validating the neural network approximations.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Import the exact Gaussian solver
from experiments.gaussian.solver import GaussianMarginalSolver

## 1. Define Gaussian Marginals

We define three Gaussian distributions with different means and covariances.

In [ ]:
# 1D Example: Three Gaussians
dim = 1

# Source: N(0, 1)
mu0 = np.array([0.0])
cov0 = np.array([[1.0]])

# Intermediate at t=0.5: N(3, 0.25)
mu_half = np.array([3.0])
cov_half = np.array([[0.25]])

# Target: N(1, 2)
mu1 = np.array([1.0])
cov1 = np.array([[2.0]])

print("Marginals:")
print(f"  t=0.0: N({mu0[0]}, {cov0[0,0]})")
print(f"  t=0.5: N({mu_half[0]}, {cov_half[0,0]})")
print(f"  t=1.0: N({mu1[0]}, {cov1[0,0]})")

## 2. Create Solver and Compute Exact Solution

In [ ]:
# Create solver with intermediate marginal
solver = GaussianMarginalSolver(
    d=dim,
    source_mean=mu0,
    source_cov=cov0,
    target_mean=mu1,
    target_cov=cov1,
    tks=[0.5],
    marginal_means=[mu_half],
    marginal_covs=[cov_half],
)

# Solve using shooting method
success = solver.solve()
print(f"Solver converged: {success}")

## 3. Sample Exact Trajectories

In [ ]:
# Sample initial conditions from source
n_samples = 100
x0 = np.random.randn(n_samples, dim) * np.sqrt(cov0[0, 0]) + mu0

# Generate exact trajectories
t_eval = np.linspace(0, 1, 50)
trajectories = solver.sample_trajectories(x0, t_eval)

print(f"Trajectory shape: {trajectories.shape}")

## 4. Visualize Exact Solution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Trajectories
ax = axes[0]
for i in range(n_samples):
    ax.plot(t_eval, trajectories[:, i, 0], alpha=0.3, lw=0.5, color='purple')

# Plot marginal means
ax.axhline(mu0[0], color='red', ls='--', label='Source mean')
ax.axhline(mu_half[0], color='orange', ls='--', label='Intermediate mean')
ax.axhline(mu1[0], color='green', ls='--', label='Target mean')

ax.axvline(0.5, color='gray', ls=':', alpha=0.5)
ax.set_xlabel('Time t')
ax.set_ylabel('x(t)')
ax.set_title('Exact Gaussian Trajectories')
ax.legend()

# Panel 2: Mean and covariance evolution
ax = axes[1]

# Compute mean and std at each time
mean_t = np.mean(trajectories[:, :, 0], axis=1)
std_t = np.std(trajectories[:, :, 0], axis=1)

ax.plot(t_eval, mean_t, 'b-', lw=2, label='Mean')
ax.fill_between(t_eval, mean_t - std_t, mean_t + std_t, alpha=0.3, label='±1 std')

# Mark target values
ax.scatter([0], [mu0[0]], s=100, c='red', zorder=5)
ax.scatter([0.5], [mu_half[0]], s=100, c='orange', zorder=5)
ax.scatter([1], [mu1[0]], s=100, c='green', zorder=5)

ax.set_xlabel('Time t')
ax.set_ylabel('x(t)')
ax.set_title('Mean and Variance Evolution')
ax.legend()

plt.tight_layout()
plt.show()

## 5. Compare Learned vs Exact

In [ ]:
from collections import OrderedDict
from otpfm import OTPFM
from otpfm.potentials import W2InfPotential

# Create and train a neural network model
potentials = OrderedDict({
    0.5: W2InfPotential(tk=0.5, strength=100.0, lambda_fn_type='gaussian', width=0.1)
})

model = OTPFM(
    d=dim,
    tks=[0.5],
    potentials=potentials,
    flownet_args={'hidden_dim': 64, 'num_hidden_layers': 2}
)

# Quick training loop
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_samples_train = 1000

for epoch in range(200):
    model.train()
    
    # Sample from marginals
    x0_batch = torch.randn(64, dim) * np.sqrt(cov0[0, 0]) + mu0[0]
    x_half_batch = torch.randn(64, dim) * np.sqrt(cov_half[0, 0]) + mu_half[0]
    x1_batch = torch.randn(64, dim) * np.sqrt(cov1[0, 0]) + mu1[0]
    
    xs = torch.stack([x0_batch, x_half_batch, x1_batch], dim=1)
    otp_alpha = 1.0 / (1.0 + np.exp(-6.0 * (epoch - 100) / 100))
    
    loss = model.forward_with_losses(xs, otp_alpha, do_otp=True)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.update_ema()

print("Training complete")

In [ ]:
# Sample from learned model
model.eval()
x0_test = torch.randn(n_samples, dim) * np.sqrt(cov0[0, 0]) + mu0[0]

with torch.no_grad():
    learned_traj, t_learned = model.sample(x0_test, n_steps=50, ema=True)

learned_traj = learned_traj.numpy()
t_learned = t_learned.numpy()

# Compute exact trajectories with same initial conditions
exact_traj = solver.sample_trajectories(x0_test.numpy(), t_learned)

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Exact
ax = axes[0]
for i in range(min(50, n_samples)):
    ax.plot(t_learned, exact_traj[:, i, 0], alpha=0.3, lw=0.5, color='blue')
ax.axhline(mu0[0], color='red', ls='--')
ax.axhline(mu_half[0], color='orange', ls='--')
ax.axhline(mu1[0], color='green', ls='--')
ax.set_xlabel('Time t')
ax.set_ylabel('x(t)')
ax.set_title('Exact Solution')

# Learned
ax = axes[1]
for i in range(min(50, n_samples)):
    ax.plot(t_learned, learned_traj[:, i, 0], alpha=0.3, lw=0.5, color='purple')
ax.axhline(mu0[0], color='red', ls='--')
ax.axhline(mu_half[0], color='orange', ls='--')
ax.axhline(mu1[0], color='green', ls='--')
ax.set_xlabel('Time t')
ax.set_ylabel('x(t)')
ax.set_title('Learned Solution (OTP-FM)')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. Computing exact solutions for multi-marginal OT with Gaussian marginals
2. Training OTP-FM to approximate these solutions
3. Quantitative comparison between exact and learned trajectories

The exact solver provides ground truth for validating the neural network approximations and understanding the theoretical properties of the optimal transport problem with intermediate marginal constraints.